### Install all the neccessaary package

In [1]:
%pip install numpy
%pip install opencv-python
%pip install opencv-contrib-python

  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/7d/31/6e35a247acb1bfc19226791dfc7d4c30002cd4e620e11e58b0ddf836fe52/numpy-2.3.1-cp311-cp311-macosx_14_0_arm64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 375.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 8.4 MB/s eta 0:00:0000:0100:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Obtaining dependency information for opencv-python from https://files.pythonhosted.org/packages/05/4d/53b30a2a3ac1f75f65a59eb29cf2ee7207ce64867db47036ad61743d5a23/opencv_python-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 14.7 MB/s eta 0:00:

### Import packages and libraries

In [2]:
# A computer vision library
import cv2
# A scientific computation library
import numpy as np

### Test a camera

In [4]:

vid = cv2.VideoCapture(0)

while True:
    ret, frame = vid.read()
    
    # Add text at bottom right
    cv2.putText(frame, "Click 'q' to exit webcam", (frame.shape[1] - 300, frame.shape[0] - 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    cv2.imshow("Video", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

2025-07-07 06:28:32.799 Python[60176:1151861] WARNING: Secure coding is not enabled for restorable state! Enable secure coding by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState: and returning YES.


### Task One:

Instruct students to read videos stored on the computer

### Face Detection

This is to be able to draw a bounding box around the face.

We are also gathering data for the training of the recogition. The code will request for a name, put a name in the box provided then press Enter. A window frame will pop up, and when it sees a face it will draw a green bounding as it has detected a face. When the bounding box is not shaking and it is stable, press q and you will be prompted for a name again. The process will be repeated all over again. We set training for five(5) people

In [25]:
cap = cv2.VideoCapture(0)
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
face_data = []
labels = []
number_of_people = 2
for i in range(number_of_people):
    name = input(f"Enter the name of person {i+1}: ")
    while True:
        ret, frame = cap.read()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
        for (x, y, w, h) in faces:
            face = gray[y:y + h, x:x + w]
            face = cv2.resize(face, (512, 512), interpolation=cv2.INTER_AREA)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 4)
            cv2.putText(frame, "Click 'q' to save image", (frame.shape[1] - 300, frame.shape[0] - 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.imshow('Frame', frame)

            if 0xFF == ord('q'):
                break

        cv2.imshow('Frame', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    face_data.append(face)
    labels.append(name)
cap.release()
cv2.destroyAllWindows()


# Train on different people both through webcam and videos

### Train a face recognition model

In [26]:
# Create a LBPH face recognizer
labels2={i:j for i,j in enumerate(labels)}

print(labels2, face_data)
face_recognizer = cv2.face.LBPHFaceRecognizer_create(
        radius=2,           # Increased radius for better feature extraction
        neighbors=16,       # More neighbors for better accuracy
        grid_x=8,           # Grid size for feature extraction
        grid_y=8,
        threshold=100.0     # Threshold for recognition
)

# Train the face recognizer on the dataset
face_recognizer.train(face_data, np.array(list(labels2.keys())))

{0: 'Peace', 1: 'Abdul'} [array([[91, 89, 84, ..., 49, 43, 37],
       [93, 89, 79, ..., 54, 49, 42],
       [95, 87, 75, ..., 57, 60, 49],
       ...,
       [28, 28, 26, ..., 22, 22, 22],
       [28, 28, 26, ..., 22, 22, 22],
       [28, 28, 26, ..., 22, 22, 22]], shape=(512, 512), dtype=uint8), array([[217, 219, 217, ..., 183, 179, 186],
       [199, 205, 208, ..., 184, 179, 193],
       [213, 212, 210, ..., 185, 178, 181],
       ...,
       [180, 179, 179, ..., 224, 160, 181],
       [180, 179, 180, ..., 226, 143, 181],
       [179, 179, 180, ..., 221, 140, 181]], shape=(512, 512), dtype=uint8)]


### Face Recognition:

This part of the code is to be able to detect the person that was stored. The code will detect the face of anyone stored and it will be unknown if the person is not part of the data. Press q when done.

In [ ]:

# Real-time face recognition from webcam
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    for (x, y, w, h) in faces:
        face_roi = gray[y:y+h, x:x+w]
        face_features = cv2.resize(face_roi, (512, 512), interpolation=cv2.INTER_AREA)
        label, confidence = face_recognizer.predict(face_features)
        if confidence > 50 and confidence <= 100:
            detection_name = labels2[abs(label)]
            color = (0, 255, 0)  # Green for recognized
        else:
            detection_name = "Unknown"
            color = (0, 0, 255)  # Red for unknown
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, str(detection_name), (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)

    cv2.imshow('Face Recognition', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
0 83.26987486671709
0 71.0831190343944
0 83.99973438261968
0 59.32464751133889
-1 1.7976931348623157e+308
0 59.7185960251587
0 56.29343406689924
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
0 57.59378449169129
0 60.32559558886368
0 57.12983522016753
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
0 55.295098363029865
0 50.95229649559873
0 55.623154026629635
0 58.665781698667395
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.7976931348623157e+308
-1 1.79769

### Task Three:

Change the bounding box color, on the video stream to any color of your choice

### Task Three:

Register 5 of your friends and train the model to recognize thier faces

### Task Four:

Save models using cv.save()

### Take home task:

 Take home task, create a facial authentication system on the web using streamlit

[ ] Register face(s): Sign in

[ ] Login using Face registered